In [ ]:
!pip3 install pyspark

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql import types as T
from pyspark.sql import window as W
from pyspark.sql import functions as F

spark = SparkSession.builder \
        .master("local") \
        .appName("Colab") \
        .getOrCreate()

#1번

. member 테이블에서 회원 상태별 인원수를 내림차순으로 보여주세요. (6

In [ ]:
df_member=spark.read.parquet('./member.parquet',header=True)
df_member.show()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
|100010| 남|유료회원|초4학년|
|100011| 여|유료회원|초4학년|
|100017| 여|유료회원|  0학년|
|100019| 남|유료회원|초3학년|
| 10002| 여|유료회원|초4학년|
|100020| 남|유료회원|초5학년|
| 10003| 남|유료회원|초3학년|
|100030| 남|유료회원|  0학년|
|100031| 여|유료회원|초2학년|
|100037| 남|유료회원|초3학년|
|100039| 남|유료회원|초4학년|
| 10004| 여|유료회원|초3학년|
|100042| 남|유료회원|초2학년|
|100044| 여|유료회원|초5학년|
|100045| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 20 rows



In [ ]:
df_member.dtypes

[('idx', 'string'),
 ('sex', 'string'),
 ('status', 'string'),
 ('grade', 'string')]

In [ ]:
# df_member.orderBy(F.col("status").desc()).show()
# df_member.groupBy("status") \
#          .count() \
#          .orderBy(F.desc("count")) \
#          .show()
df_member.groupby('status').count().orderBy(F.desc("count")).show()

+--------+-----+
|  status|count|
+--------+-----+
|유료회원|66032|
|학습만료|   92|
|    신규|   75|
|  재구매|   70|
|    이월|   22|
|    복회|    7|
|    취소|    6|
+--------+-----+



#2번

 study_his 테이블의 pointnm 컬럼에 대해 공백을 모두 없애주세요 (6)

In [ ]:
df_study=spark.read.parquet('./study_his.parquet',header=True)
df_study.show()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
|89542| 202305|20230509| 중학 3학년|
|23940| 202305|20230516| 중학 1학년|
|23940| 202305|20230516| 중학 1학년|
|40502| 202304|20230412| 중학 1학년|
| 1741| 202304|20230405| 중학 1학년|
|50161| 202304|20230428| 중학 1학년|
+-----+-------+--------+-----------+
only showing top 20 rows



In [ ]:
df_study.dtypes

[('idx', 'string'),
 ('proc_ym', 'string'),
 ('proc_ymd', 'string'),
 ('pointnm', 'string')]

In [ ]:
var = "Data Science"

def str_udf(var):
  #공백을+로 치환
  var = var.replace(" ", "+")
  return var
str_udf(var)

'Data+Science'

In [ ]:
user_udf = F.udf(str_udf,returnType = T.StringType())
df_study.withColumn("new_pointnm", user_udf(F.col("pointnm"))).drop("pointnm").show()


+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|new_pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글+스피치|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
| 8604| 202306|20230606| 중학+3학년|
|89542| 202305|20230509| 중학+3학년|
|23940| 202305|20230516| 중학+1학년|
|23940| 202305|20230516| 중학+1학년|
|40502| 202304|20230412| 중학+1학년|
| 1741| 202304|20230405| 중학+1학년|
|50161| 202304|20230428| 중학+1학년|
+-----+-------+--------+-----------+
only showing top 20 rows



# 3번


point_his의 proc_ymd 컬럼의 날짜 표현 형식을 yyyy-mm-dd 형식이 되도록 바꿔주세요. (6)(UDF, slicing 사용 X)

In [ ]:
df_point=spark.read.parquet('./point_his.parquet',header=True)
df_point.show()

+-----+-------+--------+-----+
|  idx|proc_ym|proc_ymd|point|
+-----+-------+--------+-----+
|96465| 202306|20230624| 1000|
|96465| 202306|20230624|  500|
|87940| 202304|20230405| 2000|
|87940| 202304|20230405| 3500|
|87940| 202304|20230405| 4000|
|87940| 202304|20230405| 3000|
|87940| 202304|20230405| 2500|
|88058| 202304|20230405| 3000|
|88058| 202304|20230405| 1000|
|88058| 202304|20230405| 1500|
|88058| 202304|20230405| 2500|
|88058| 202304|20230405|  500|
|88058| 202304|20230405| 2000|
|95844| 202306|20230617| 1500|
|95844| 202306|20230617| 1000|
|95050| 202306|20230612| 3500|
|95050| 202306|20230612| 4000|
|95050| 202306|20230612| 4500|
|92560| 202305|20230511| 1000|
|92560| 202305|20230511|  500|
+-----+-------+--------+-----+
only showing top 20 rows



In [ ]:
df_point.withColumn("new_point", F.to_date(F.col("proc_ymd"), "yyyyMMdd")).show()

+-----+-------+--------+-----+----------+
|  idx|proc_ym|proc_ymd|point| new_point|
+-----+-------+--------+-----+----------+
|96465| 202306|20230624| 1000|2023-06-24|
|96465| 202306|20230624|  500|2023-06-24|
|87940| 202304|20230405| 2000|2023-04-05|
|87940| 202304|20230405| 3500|2023-04-05|
|87940| 202304|20230405| 4000|2023-04-05|
|87940| 202304|20230405| 3000|2023-04-05|
|87940| 202304|20230405| 2500|2023-04-05|
|88058| 202304|20230405| 3000|2023-04-05|
|88058| 202304|20230405| 1000|2023-04-05|
|88058| 202304|20230405| 1500|2023-04-05|
|88058| 202304|20230405| 2500|2023-04-05|
|88058| 202304|20230405|  500|2023-04-05|
|88058| 202304|20230405| 2000|2023-04-05|
|95844| 202306|20230617| 1500|2023-06-17|
|95844| 202306|20230617| 1000|2023-06-17|
|95050| 202306|20230612| 3500|2023-06-12|
|95050| 202306|20230612| 4000|2023-06-12|
|95050| 202306|20230612| 4500|2023-06-12|
|92560| 202305|20230511| 1000|2023-05-11|
|92560| 202305|20230511|  500|2023-05-11|
+-----+-------+--------+-----+----

# 4번

학습만료 회원들이 공부한 과목 리스트를 TOP3를 뽑아주세요 (8)

In [ ]:
df_member.show()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
|100010| 남|유료회원|초4학년|
|100011| 여|유료회원|초4학년|
|100017| 여|유료회원|  0학년|
|100019| 남|유료회원|초3학년|
| 10002| 여|유료회원|초4학년|
|100020| 남|유료회원|초5학년|
| 10003| 남|유료회원|초3학년|
|100030| 남|유료회원|  0학년|
|100031| 여|유료회원|초2학년|
|100037| 남|유료회원|초3학년|
|100039| 남|유료회원|초4학년|
| 10004| 여|유료회원|초3학년|
|100042| 남|유료회원|초2학년|
|100044| 여|유료회원|초5학년|
|100045| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 20 rows



In [ ]:
df_study.show()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
|89542| 202305|20230509| 중학 3학년|
|23940| 202305|20230516| 중학 1학년|
|23940| 202305|20230516| 중학 1학년|
|40502| 202304|20230412| 중학 1학년|
| 1741| 202304|20230405| 중학 1학년|
|50161| 202304|20230428| 중학 1학년|
+-----+-------+--------+-----------+
only showing top 20 rows



In [ ]:
df_memeber_point=df_member.join(df_study,df_member.idx == df_study.idx,"inner")
df_memeber_point.filter(F.col("status") == "학습만료").groupby("pointnm").agg(F.count(F.col("pointnm")).alias("pointnm_cnt")).orderBy(F.desc("pointnm_cnt")).show(3)

+------------+-----------+
|     pointnm|pointnm_cnt|
+------------+-----------+
|  중학 1학년|        122|
|  중학 2학년|         39|
|AI서술형평가|         39|
+------------+-----------+
only showing top 3 rows



# 5번

5. member_dup 테이블에 학년이 여러개인 idx가 있습니다. 해당 idx의 학년 중 가장 높은 학년만 남겨서 idx와 grade가 1:1 대응이 되도록 만들어주세요

In [ ]:
df_member_dup=spark.read.parquet('./member_dup.parquet',header=True)
df_member_dup.show()

+-----+---+--------+-------+
|  idx|sex|  status|  grade|
+-----+---+--------+-------+
| 6884| 여|유료회원|초3학년|
| 6331| 남|유료회원|초3학년|
|69294| 남|유료회원|초5학년|
|31531| 여|유료회원|초1학년|
|85784| 여|유료회원|초2학년|
|58058| 여|유료회원|초3학년|
|  777| 남|유료회원|초3학년|
| 5482| 여|유료회원|초2학년|
|63447| 남|유료회원|초2학년|
|54957| 남|유료회원|초4학년|
|55340| 남|유료회원|초2학년|
|72887| 여|유료회원|초3학년|
|40008| 여|유료회원|초3학년|
|57551| 남|유료회원|초5학년|
|58583| 여|유료회원|초3학년|
|32219| 남|유료회원|초1학년|
|13136| 여|유료회원|초4학년|
|22122| 남|유료회원|초6학년|
|18306| 남|유료회원|초4학년|
|28057| 남|유료회원|초3학년|
+-----+---+--------+-------+
only showing top 20 rows



In [ ]:
df_member_dup.filter(F.col("grade").like("%6%")).dropDuplicates(subset=["idx"]).show()

+------+--------+--------+-------+
|   idx|     sex|  status|  grade|
+------+--------+--------+-------+
| 10000|      여|유료회원|초6학년|
|100056|      남|유료회원|초6학년|
|100059|      여|유료회원|초6학년|
|100062|      남|유료회원|초6학년|
|100087|      여|유료회원|초6학년|
| 10017|정보없음|유료회원|초6학년|
|100177|      여|유료회원|초6학년|
| 10018|      남|유료회원|초6학년|
| 10019|      남|유료회원|초6학년|
|  1002|      남|유료회원|초6학년|
| 10020|      여|유료회원|초6학년|
|100222|      남|유료회원|초6학년|
| 10023|      여|유료회원|초6학년|
|100252|      여|유료회원|초6학년|
| 10030|      여|유료회원|초6학년|
| 10032|      남|유료회원|초6학년|
|100367|      여|유료회원|초6학년|
| 10037|      남|유료회원|초6학년|
|100386|      남|유료회원|초6학년|
|100397|      여|유료회원|초6학년|
+------+--------+--------+-------+
only showing top 20 rows



#6번

등록일(reg_date)이 2023.03.15일인 유료회원들이 가장 많이 착용한 아이템(codename) TOP3를 뽑아주세요

In [ ]:
df_reg=spark.read.parquet('./regdate.parquet',header=True)
df_reg.show()

+---+--------+
|idx| regdate|
+---+--------+
|  1|20221206|
|  2|20221206|
|  3|20221206|
|  4|20221206|
|  5|20221206|
|  7|20221206|
|  8|20221206|
|  9|20221206|
| 10|20221206|
| 11|20221206|
| 12|20221206|
| 13|20221206|
| 14|20221206|
| 15|20221206|
| 16|20221206|
| 17|20221206|
| 18|20221206|
| 19|20221206|
| 20|20221206|
| 21|20221206|
+---+--------+
only showing top 20 rows



In [ ]:
df_item=spark.read.parquet('./item_his.parquet',header=True)
df_item.show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|76257|190.0| 20230

In [ ]:
# df_reg_item_join=df_reg.join(df_item, df_reg.idx== df_item.idx,"inner")
# df_reg_item_join.show()
df_reg_item_join=df_reg.join(df_item, ['idx'])
df_reg_item_join.show()

+-----+--------+-----+-------+--------+----------+--------------+-----+
|  idx| regdate|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+--------+-----+-------+--------+----------+--------------+-----+
|53687|20221215|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|20221215|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|20221206|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572|20230407| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572|20230407| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|20221207|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|
|26112|20221207|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|20221213|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|20221213|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|20221213|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|20221213|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|20221213|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822|20221208| 39.0| 202306|20230608|   

In [ ]:
df_final_join=df_reg_item_join.join(df_member,['idx'])
df_final_join.show()

+-----+--------+-----+-------+--------+----------+--------------+-----+--------+--------+-------+
|  idx| regdate|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|     sex|  status|  grade|
+-----+--------+-----+-------+--------+----------+--------------+-----+--------+--------+-------+
|53687|20221215|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|      남|유료회원|초1학년|
|53687|20221215|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|      남|유료회원|초1학년|
|20163|20221206|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|      여|유료회원|초3학년|
|88572|20230407| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|정보없음|유료회원|초3학년|
|88572|20230407| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|정보없음|유료회원|초3학년|
|26112|20221207|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|      남|유료회원|초5학년|
|26112|20221207|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|      남|유료회원|초5학년|
|49541|20221213|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|      남|유료회원|초3학년|
|49541|20221213|170.0| 202306|20230630|      신발|아바타파츠구분|  150|      남|유료회원|초

In [ ]:
# df_final_join.filter((F.col("regdate") == "20230315") | (F.col("status") == "유료회원")).show()
# item.filter((F.col("codename") == "액세서리") | (F.col("price")<50)).show()
df_final_join.filter(F.col("regdate") == "20230315").filter(F.col("status") == "유료회원").show()

+-----+--------+-----+-------+--------+----------+--------------+-----+---+--------+-------+
|  idx| regdate|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|sex|  status|  grade|
+-----+--------+-----+-------+--------+----------+--------------+-----+---+--------+-------+
|84801|20230315|112.0| 202306|20230627|      헤어|아바타파츠구분|  150| 여|유료회원|초2학년|
|84801|20230315|112.0| 202306|20230627|      하의|아바타파츠구분|  250| 여|유료회원|초2학년|
|84801|20230315|112.0| 202306|20230627|      얼굴|아바타파츠구분|  150| 여|유료회원|초2학년|
|84757|20230315|101.0| 202305|20230531|      얼굴|아바타파츠구분|  150| 남|유료회원|초1학년|
|84757|20230315|101.0| 202305|20230531|      얼굴|아바타파츠구분|  150| 남|유료회원|초1학년|
|84757|20230315|101.0| 202305|20230531|상태메시지|  기타파츠구분|   50| 남|유료회원|초1학년|
|84757|20230315|101.0| 202305|20230531|상태메시지|  기타파츠구분|   50| 남|유료회원|초1학년|
|84757|20230315|101.0| 202305|20230531|      헤어|아바타파츠구분|  150| 남|유료회원|초1학년|
|84829|20230315| 94.0| 202304|20230420|      헤어|아바타파츠구분|  150| 여|유료회원|초1학년|
|84829|20230315| 94.0| 202304|20230420|  

In [ ]:
df_final_join.filter(F.col("regdate") == "20230315").filter(F.col("status") == "유료회원").groupby("codename").agg(F.count(F.col("codename")).alias("codename_cnt")).orderBy(F.desc("codename_cnt")).show(3)

+----------+------------+
|  codename|codename_cnt|
+----------+------------+
|상태메시지|          74|
|      헤어|          67|
|      얼굴|          47|
+----------+------------+
only showing top 3 rows



#7번

member 테이블의 grade 컬럼에서 숫자만 뽑아 grade라는 컬럼을 재구성해주세요. (초3학년 -> 3) (8)  (UDF, slicing 사용 X)


In [ ]:
df_member.show()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
|100010| 남|유료회원|초4학년|
|100011| 여|유료회원|초4학년|
|100017| 여|유료회원|  0학년|
|100019| 남|유료회원|초3학년|
| 10002| 여|유료회원|초4학년|
|100020| 남|유료회원|초5학년|
| 10003| 남|유료회원|초3학년|
|100030| 남|유료회원|  0학년|
|100031| 여|유료회원|초2학년|
|100037| 남|유료회원|초3학년|
|100039| 남|유료회원|초4학년|
| 10004| 여|유료회원|초3학년|
|100042| 남|유료회원|초2학년|
|100044| 여|유료회원|초5학년|
|100045| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 20 rows



In [ ]:
# 숫자 부분 추출
df_member_new = df_member.withColumn("grade", F.regexp_extract("grade", r"(\d+)", 1))
df_member_new.show()

+------+---+--------+-----+
|   idx|sex|  status|grade|
+------+---+--------+-----+
|   100| 남|유료회원|    1|
|  1000| 여|유료회원|    5|
| 10000| 여|유료회원|    6|
|100007| 남|유료회원|    4|
| 10001| 남|유료회원|    2|
|100010| 남|유료회원|    4|
|100011| 여|유료회원|    4|
|100017| 여|유료회원|    0|
|100019| 남|유료회원|    3|
| 10002| 여|유료회원|    4|
|100020| 남|유료회원|    5|
| 10003| 남|유료회원|    3|
|100030| 남|유료회원|    0|
|100031| 여|유료회원|    2|
|100037| 남|유료회원|    3|
|100039| 남|유료회원|    4|
| 10004| 여|유료회원|    3|
|100042| 남|유료회원|    2|
|100044| 여|유료회원|    5|
|100045| 남|유료회원|    2|
+------+---+--------+-----+
only showing top 20 rows



# 8번

study_his 월별로 pointnm 컬럼에 대한 point 발생 수(count)를 오름차순으로 순번 매겨주세요

In [ ]:
df_study.show()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
|89542| 202305|20230509| 중학 3학년|
|23940| 202305|20230516| 중학 1학년|
|23940| 202305|20230516| 중학 1학년|
|40502| 202304|20230412| 중학 1학년|
| 1741| 202304|20230405| 중학 1학년|
|50161| 202304|20230428| 중학 1학년|
+-----+-------+--------+-----------+
only showing top 20 rows



In [ ]:
df_point.show()

+-----+-------+--------+-----+
|  idx|proc_ym|proc_ymd|point|
+-----+-------+--------+-----+
|96465| 202306|20230624| 1000|
|96465| 202306|20230624|  500|
|87940| 202304|20230405| 2000|
|87940| 202304|20230405| 3500|
|87940| 202304|20230405| 4000|
|87940| 202304|20230405| 3000|
|87940| 202304|20230405| 2500|
|88058| 202304|20230405| 3000|
|88058| 202304|20230405| 1000|
|88058| 202304|20230405| 1500|
|88058| 202304|20230405| 2500|
|88058| 202304|20230405|  500|
|88058| 202304|20230405| 2000|
|95844| 202306|20230617| 1500|
|95844| 202306|20230617| 1000|
|95050| 202306|20230612| 3500|
|95050| 202306|20230612| 4000|
|95050| 202306|20230612| 4500|
|92560| 202305|20230511| 1000|
|92560| 202305|20230511|  500|
+-----+-------+--------+-----+
only showing top 20 rows



In [ ]:
from typing import cast
df_join_study_point=df_study.join(df_point,['proc_ym'])
df_join_study_point.show()
df_join_study_point=df_join_study_point.withColumn("point",F.col("point").cast(T.IntegerType()))
df_join_study_point.dtypes

+-------+-----+--------+-----------+-----+--------+-----+
|proc_ym|  idx|proc_ymd|    pointnm|  idx|proc_ymd|point|
+-------+-----+--------+-----------+-----+--------+-----+
| 202306|88311|20230628|한글 스피치|96252|20230622| 2000|
| 202306|88311|20230628|한글 스피치|96252|20230622| 1500|
| 202306|88311|20230628|한글 스피치|96252|20230622| 1000|
| 202306|88311|20230628|한글 스피치|96252|20230622| 2500|
| 202306|88311|20230628|한글 스피치|89063|20230619|  500|
| 202306|88311|20230628|한글 스피치|95280|20230611| 5000|
| 202306|88311|20230628|한글 스피치|95280|20230611| 3500|
| 202306|88311|20230628|한글 스피치|95280|20230611| 4500|
| 202306|88311|20230628|한글 스피치|95280|20230611| 4000|
| 202306|88311|20230628|한글 스피치|96332|20230623| 2500|
| 202306|88311|20230628|한글 스피치|96332|20230623| 3000|
| 202306|88311|20230628|한글 스피치|96332|20230623| 1500|
| 202306|88311|20230628|한글 스피치|96332|20230623| 1000|
| 202306|88311|20230628|한글 스피치|96332|20230623| 2000|
| 202306|88311|20230628|한글 스피치|96332|20230623| 3500|
| 202306|88311|20230628|한글 스피치|

[('proc_ym', 'string'),
 ('idx', 'string'),
 ('proc_ymd', 'string'),
 ('pointnm', 'string'),
 ('idx', 'string'),
 ('proc_ymd', 'string'),
 ('point', 'int')]

In [ ]:
df_1 = df_join_study_point.groupby('proc_ym','pointnm','point').agg(F.count(F.col("point")).alias("point_cnt"))
df_2 =df_1.withColumn("point_cnt",F.col("point_cnt").cast(T.IntegerType()))
df_3 = df_2.orderBy("point_cnt",ascending=True)
df_3.show()

+-------+---------+-----+---------+
|proc_ym|  pointnm|point|point_cnt|
+-------+---------+-----+---------+
| 202305|학교 체험| 5000|    20195|
| 202305|학교 체험| 4500|    20244|
| 202305|학교 체험| 3500|    20370|
| 202305|학교 체험| 4000|    20440|
| 202305|학교 체험| 3000|    20531|
| 202305|학교 체험| 2500|    20580|
| 202305|학교 체험| 2000|    20706|
| 202305|학교 체험| 1500|    20818|
| 202305|학교 체험| 1000|    20951|
| 202305|학교 체험|  500|    21112|
| 202306|학교 체험| 5000|    41446|
| 202306|학교 체험| 4500|    41701|
| 202306|학교 체험| 4000|    41990|
| 202306|학교 체험| 3500|    42228|
| 202306|학교 체험| 3000|    42415|
| 202306|학교 체험| 2500|    42670|
| 202306|학교 체험| 2000|    42925|
| 202306|학교 체험| 1500|    43299|
| 202306|학교 체험| 1000|    43452|
| 202306|학교 체험|  500|    43741|
+-------+---------+-----+---------+
only showing top 20 rows



#9번

레벨이 151~160 에 있는 유저들 중 딱 한 명씩만 등록한 날짜들을 구해주세요.
   레벨은 idx가 가진 레벨중 가장 높은 레벨로 사용(10) -> 중복처리에 유의
   

In [ ]:
df_item.show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|76257|190.0| 20230

In [ ]:
df_reg.show()

+---+--------+
|idx| regdate|
+---+--------+
|  1|20221206|
|  2|20221206|
|  3|20221206|
|  4|20221206|
|  5|20221206|
|  7|20221206|
|  8|20221206|
|  9|20221206|
| 10|20221206|
| 11|20221206|
| 12|20221206|
| 13|20221206|
| 14|20221206|
| 15|20221206|
| 16|20221206|
| 17|20221206|
| 18|20221206|
| 19|20221206|
| 20|20221206|
| 21|20221206|
+---+--------+
only showing top 20 rows



In [ ]:
df_item_reg=df_item.join(df_reg,['idx'])
df_item_reg.show()

+-----+-----+-------+--------+----------+--------------+-----+--------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price| regdate|
+-----+-----+-------+--------+----------+--------------+-----+--------+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|20221215|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|20221215|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|20221206|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|20230407|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|20230407|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|20221207|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|20221207|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|20221213|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|20221213|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|20221213|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|20221213|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|20221213|
|37822| 39.0| 202306|20230608|      헤어|아바타

In [ ]:
df_item_reg.filter((F.col("lv") >= 151) &  (F.col("lv")<161)).dropDuplicates(subset=['idx']).show()
# item.dropDuplicates(subset=["proc_ym","codename"]).show()

+-----+-----+-------+--------+----------+--------------+-----+--------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price| regdate|
+-----+-----+-------+--------+----------+--------------+-----+--------+
|10009|159.0| 202304|20230414|      헤어|아바타파츠구분|  150|20221206|
|10037|152.0| 202304|20230408|      신발|아바타파츠구분|   60|20221206|
|10038|151.0| 202304|20230404|      상의|아바타파츠구분|  200|20221206|
|10041|152.0| 202305|20230502|      신발|아바타파츠구분|   60|20221206|
|10046|154.0| 202305|20230508|      신발|아바타파츠구분|   60|20221206|
|10070|160.0| 202305|20230526|      상의|아바타파츠구분|  200|20221206|
|10073|153.0| 202305|20230505|      신발|아바타파츠구분|   60|20221206|
|10085|158.0| 202305|20230502|      헤어|아바타파츠구분|  150|20221206|
|10089|153.0| 202304|20230405|      얼굴|아바타파츠구분|  150|20221206|
| 1011|160.0| 202304|20230427|      하의|아바타파츠구분|  250|20221206|
| 1013|158.0| 202304|20230412|상태메시지|  기타파츠구분|  100|20221206|
|10139|159.0| 202304|20230426|      헤어|아바타파츠구분|  150|20221206|
|10141|158.0| 202304|20230403|

#10번

 item_his 테이블에서 레벨이 null인 유저가 가장 많은 날짜를 구해주세요 (10)

In [ ]:
df_item.show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|76257|190.0| 20230

In [ ]:
df_item.dtypes

[('idx', 'string'),
 ('lv', 'string'),
 ('proc_ym', 'string'),
 ('proc_ymd', 'string'),
 ('codename', 'string'),
 ('mascodename', 'string'),
 ('price', 'string')]

In [ ]:
from pyspark.sql.functions import col, count
df_null=df_item.filter(col("lv").isNull())
df_null.show()
df_null.groupBy("proc_ymd").count().orderBy(col("count").desc()).show(1)


+-----+----+-------+--------+--------+--------------+-----+
|  idx|  lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+----+-------+--------+--------+--------------+-----+
|92027|NULL| 202305|20230505|  코스튬|아바타파츠구분|  300|
|92131|NULL| 202305|20230507|  코스튬|아바타파츠구분|  300|
|91962|NULL| 202305|20230506|  코스튬|아바타파츠구분|  300|
|94835|NULL| 202306|20230605|  코스튬|아바타파츠구분|  300|
|94835|NULL| 202306|20230605|  코스튬|아바타파츠구분|  300|
|91129|NULL| 202304|20230427|  코스튬|아바타파츠구분|  300|
|91129|NULL| 202304|20230427|  코스튬|아바타파츠구분|  300|
|91129|NULL| 202304|20230427|  코스튬|아바타파츠구분|  300|
|93173|NULL| 202305|20230517|  코스튬|아바타파츠구분|  300|
|96008|NULL| 202306|20230618|  코스튬|아바타파츠구분|  300|
|96008|NULL| 202306|20230618|  코스튬|아바타파츠구분|  300|
|96008|NULL| 202306|20230618|  코스튬|아바타파츠구분|  300|
|91835|NULL| 202305|20230503|  코스튬|아바타파츠구분|  300|
|91835|NULL| 202305|20230503|  코스튬|아바타파츠구분|  300|
|91835|NULL| 202305|20230503|  코스튬|아바타파츠구분|  300|
|91835|NULL| 202305|20230503|  코스튬|아바타파츠구분|  300|
|91835|NULL| 202305|

#11번

pointnm 별로 획득한 point의 종류가 언더바(_)로 이어져서 보이도록 테이블을 만들어 주세요 (10)

In [ ]:
df_study.show()
df_point.show()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
|89542| 202305|20230509| 중학 3학년|
|23940| 202305|20230516| 중학 1학년|
|23940| 202305|20230516| 중학 1학년|
|40502| 202304|20230412| 중학 1학년|
| 1741| 202304|20230405| 중학 1학년|
|50161| 202304|20230428| 중학 1학년|
+-----+-------+--------+-----------+
only showing top 20 rows

+-----+-------+--------+-----+
|  idx|proc_ym|proc_ymd|point|
+-----+-------+--------+-----+
|96465| 202306|20230624| 1000|
|96465| 202306|20230624|  500|
|87940| 2023

In [ ]:
df_10=df_study.join(df_point,['idx'])
df_10.show()

+-----+-------+--------+-----------+-------+--------+-----+
|  idx|proc_ym|proc_ymd|    pointnm|proc_ym|proc_ymd|point|
+-----+-------+--------+-----------+-------+--------+-----+
|88311| 202306|20230628|한글 스피치| 202304|20230408| 5000|
|88311| 202306|20230628|한글 스피치| 202304|20230406|  500|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 3000|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 2500|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 1000|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 4500|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 3500|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 2000|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 1500|
|88311| 202306|20230628|한글 스피치| 202304|20230406| 4000|
|89542| 202305|20230509| 중학 3학년| 202304|20230414|  500|
|89542| 202305|20230509| 중학 3학년| 202304|20230414| 1500|
|89542| 202305|20230509| 중학 3학년| 202304|20230414| 1000|
|89542| 202305|20230509| 중학 3학년| 202304|20230415| 4000|
|89542| 202305|20230509| 중학 3학년| 202304|202304

In [ ]:
from pyspark.sql.functions import concat_ws

# pointnm 별로 획득한 point 종류를 언더바로 이어서 합치기
df_10_aggregated = df_10.groupby("pointnm").agg(concat_ws("_", F.collect_set("point")).alias("point종류"))

# 결과 출력
df_10_aggregated.show()

+-----------------+--------------------+
|          pointnm|           point종류|
+-----------------+--------------------+
|        일기 쓰기|1000_4000_2000_50...|
|        학교 체험|1000_4000_2000_50...|
|        중학 특강|4000_1000_3500_50...|
|          AI 영어|1000_4000_3500_20...|
|도전! AI 받아쓰기|1000_4000_5000_20...|
| 학교 공부 맛보기|1000_4000_3500_20...|
|      중학 신입생|1000_4000_2000_35...|
|          AI 국어|4000_1000_5000_35...|
| 밀크T 지오그래픽|4000_1000_3500_50...|
|       중학 1학년|1000_4000_3500_20...|
|      한글 스피치|1000_4000_5000_35...|
|         받아쓰기|1000_4000_2000_35...|
|       중학 3학년|1000_4000_5000_20...|
|       중학 2학년|4000_1000_5000_35...|
|   AI 구구단 게임|4000_1000_3500_50...|
|     자기소개하기|4000_1000_5000_35...|
|     AI서술형평가|4000_1000_3500_50...|
+-----------------+--------------------+



#12번

member의 status 컬럼에서 '회원'단어가 있으면 띄어쓰기하고 없으면 띄어쓰기 후 회원 단어 추가해주세요 (유료회원 -> 유효 회원 / 신규 -> 신규 회원) (10)

In [ ]:
df_member.show()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
|100010| 남|유료회원|초4학년|
|100011| 여|유료회원|초4학년|
|100017| 여|유료회원|  0학년|
|100019| 남|유료회원|초3학년|
| 10002| 여|유료회원|초4학년|
|100020| 남|유료회원|초5학년|
| 10003| 남|유료회원|초3학년|
|100030| 남|유료회원|  0학년|
|100031| 여|유료회원|초2학년|
|100037| 남|유료회원|초3학년|
|100039| 남|유료회원|초4학년|
| 10004| 여|유료회원|초3학년|
|100042| 남|유료회원|초2학년|
|100044| 여|유료회원|초5학년|
|100045| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 20 rows



In [ ]:
from pyspark.sql.functions import when, col, concat,lit
df_member_new = df_member.withColumn("new_status", when(col("status").contains("회원"), "유료 회원").otherwise(concat(col("status"), lit(" 회원"))))
df_member_new.show()
df_member_new.filter( ~(col("status") == "유료회원") ).show()
#new_status 컬럼의 값이 최종 값 입니다.

+------+---+--------+-------+----------+
|   idx|sex|  status|  grade|new_status|
+------+---+--------+-------+----------+
|   100| 남|유료회원|초1학년| 유료 회원|
|  1000| 여|유료회원|초5학년| 유료 회원|
| 10000| 여|유료회원|초6학년| 유료 회원|
|100007| 남|유료회원|초4학년| 유료 회원|
| 10001| 남|유료회원|초2학년| 유료 회원|
|100010| 남|유료회원|초4학년| 유료 회원|
|100011| 여|유료회원|초4학년| 유료 회원|
|100017| 여|유료회원|  0학년| 유료 회원|
|100019| 남|유료회원|초3학년| 유료 회원|
| 10002| 여|유료회원|초4학년| 유료 회원|
|100020| 남|유료회원|초5학년| 유료 회원|
| 10003| 남|유료회원|초3학년| 유료 회원|
|100030| 남|유료회원|  0학년| 유료 회원|
|100031| 여|유료회원|초2학년| 유료 회원|
|100037| 남|유료회원|초3학년| 유료 회원|
|100039| 남|유료회원|초4학년| 유료 회원|
| 10004| 여|유료회원|초3학년| 유료 회원|
|100042| 남|유료회원|초2학년| 유료 회원|
|100044| 여|유료회원|초5학년| 유료 회원|
|100045| 남|유료회원|초2학년| 유료 회원|
+------+---+--------+-------+----------+
only showing top 20 rows

+------+--------+--------+-------+-------------+
|   idx|     sex|  status|  grade|   new_status|
+------+--------+--------+-------+-------------+
| 10013|      남|학습만료|초3학년|학습만료 회원|
| 10096|      여|  재구매|초5학년|  재구매 회원|
|102094| 